# 03_repeat_seeds.ipynb

Chạy lặp Swin-Tiny, ResNet-50 và ConvNeXt-Tiny với seed 43 và 44, cùng giao thức với lần chạy seed 42: train, chọn checkpoint bằng macro F1 validation, rồi đo F1 sạch và mức giảm dưới năm suy giảm cố định trên validation (cùng quy trình `03_select_reference_model.ipynb`). Cuối cùng gộp với seed 42 thành bảng trung bình và độ lệch chuẩn theo model. Notebook chạy trên Kaggle với GPU. Tập test không mở.

Danh sách seed chốt trước khi chạy: 42, 43, 44. Không bỏ seed nào theo kết quả. ResNet-50 được chạy lặp vì là ứng viên sát nút của ConvNeXt-Tiny khi chọn reference (F1 sạch chênh 0,0055). MobileNetV3-Small không chạy lặp vì đã bị loại khỏi nhóm ứng viên ở bước F1 sạch (kém 0,048), xem `reports/baselines/README.md`. Thời gian ước tính trên Tesla T4 (theo thời gian các lần chạy seed 42): Swin-Tiny khoảng 14 phút mỗi lần, ResNet-50 khoảng 29 phút, ConvNeXt-Tiny khoảng 52 phút, tổng khoảng 3 đến 3,5 giờ.

## 1. Môi trường

Internet để On. Accelerator chọn GPU. Dataset input là `cthng123/durian-leaf-disease-dataset`. Phần train không cần checkpoint cũ.

Seed 42 để tổng hợp lấy từ `reports/baselines/reference_selection_with_swin.json` trong repo (nhánh `baseline-Swin-Tiny`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "baseline-Swin-Tiny"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Cấu hình

Mỗi file `configs/train_<model>_seed<N>.yaml` giống file gốc, chỉ thêm khóa `seed` ghi đè seed trong `configs/pipeline.yaml`. Split dữ liệu và seed của suy giảm không đổi, nên mọi model được chấm trên đúng cùng ảnh. Thứ tự chạy từ rẻ đến tốn: Swin-Tiny, ResNet-50, ConvNeXt-Tiny.

In [ ]:
from src.training.run import load_finetune_configs

SEEDS = [43, 44]
MODELS = ["swin_tiny", "resnet50", "convnext_tiny"]

for seed in SEEDS:
    for model in MODELS:
        train_cfg, pipeline_cfg = load_finetune_configs(f"configs/train_{model}_seed{seed}.yaml")
        assert pipeline_cfg["seed"] == seed and train_cfg["model_name"] == model
        print(f"{model} seed {pipeline_cfg['seed']}: lr {train_cfg['learning_rate']}, epoch tối đa {train_cfg['epochs']}, patience {train_cfg['patience']}")
print("Tập test: không mở")

## 3. Train

Mỗi lượt gọi `src.training.run.main`, đúng đường chạy của notebook `03_train_<model>`. Checkpoint, JSON và hình ghi vào `outputs/`; JSON và hình được chép sang `reports/baselines/<model>/` để đo độ bền ở mục 4 và để commit. Nếu phiên bị ngắt giữa chừng, chạy lại từ cell này: lượt nào đã có checkpoint thì bị bỏ qua (chỉ đúng khi cùng một phiên còn file, vì phiên mới bắt đầu từ máy sạch).

In [ ]:
import shutil

from src.training.run import ROOT, main

runs = {}
for seed in SEEDS:
    for model in MODELS:
        done = ROOT / "outputs/checkpoints" / f"{model}_seed{seed}.pt"
        if done.is_file():
            print(f"{model} seed {seed}: đã có checkpoint, bỏ qua", flush=True)
            continue
        print(f"=== {model} seed {seed} ===", flush=True)
        paths = main(f"configs/train_{model}_seed{seed}.yaml")
        runs[(model, seed)] = paths
        target = ROOT / "reports/baselines" / model
        target.mkdir(parents=True, exist_ok=True)
        for key in ("metrics", "confusion_matrix", "history"):
            shutil.copy(paths[key], target / Path(paths[key]).name)
print(sorted(runs))

## 4. Độ bền trên validation

Đo từng seed bằng `configs/reference_selection_seed<N>.yaml`: cùng validation, cùng năm suy giảm (tối 0.5, sáng 1.5, tương phản 0.5, mờ σ 2, nhiễu 0.1) và cùng seed của phép đo (42) như lần đo gốc. Không dùng để chọn lại reference model, dòng `selection` trong kết quả chỉ để tham khảo.

In [ ]:
from src.evaluation.reference import main as measure

CHECKPOINT_DIR = ROOT / "outputs/checkpoints"
records = {}
for seed in SEEDS:
    records[seed] = measure(f"configs/reference_selection_seed{seed}.yaml", CHECKPOINT_DIR)

## 5. Tổng hợp qua seed

Seed 42 lấy từ `reference_selection_with_swin.json` (cùng quy trình đo). Độ lệch chuẩn tính trên ba điểm nên chỉ là dấu hiệu, không phải ước lượng chắc.

In [ ]:
import json

import pandas as pd

base = json.loads((ROOT / "reports/baselines/reference_selection_with_swin.json").read_text(encoding="utf-8"))["models"]
sources = {42: base, **{seed: record["models"] for seed, record in records.items()}}
rows = []
for seed, models in sources.items():
    for model in MODELS:
        entry = models[model]
        rows.append({
            "model": model, "seed": seed,
            "clean_macro_f1": entry["clean"]["macro_f1"],
            **entry["corrupted"]["macro_f1"],
            "mean_corrupted": entry["corrupted"]["mean_macro_f1"],
            "degradation": entry["corrupted"]["degradation"],
        })
per_seed = pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)
display(per_seed.round(4))

summary = per_seed.groupby("model")[["clean_macro_f1", "mean_corrupted", "degradation"]].agg(["mean", "std"]).round(4)
display(summary)

pivot = per_seed.pivot(index="seed", columns="model", values="degradation")
pivot["swin_minus_convnext"] = pivot["swin_tiny"] - pivot["convnext_tiny"]
pivot["resnet_minus_convnext"] = pivot["resnet50"] - pivot["convnext_tiny"]
print("Mức giảm theo seed (âm nghĩa là model đó giảm ít hơn ConvNeXt-Tiny):")
display(pivot.round(4))

## 6. Xuất kết quả

Bảng tổng hợp ghi vào `reports/baselines/seed_repeat_summary.csv`. Checkpoint, JSON, hình và kết quả đo độ bền được nén thành hai file zip để tải về. Checkpoint tải riêng, không đưa vào Git.

In [ ]:
per_seed.to_csv(ROOT / "reports/baselines/seed_repeat_summary.csv", index=False)
shutil.make_archive("/kaggle/working/seed_runs_outputs", "zip", root_dir=ROOT, base_dir="outputs/checkpoints")
shutil.make_archive("/kaggle/working/seed_runs_reports", "zip", root_dir=ROOT, base_dir="reports/baselines")
print("Đã lưu:", sorted(path.name for path in (ROOT / "reports/baselines").glob("*seed*")))